In [ ]:
# ============================================================
# GOOGLE COLAB - HIGHWAY VEHICLE DETECTION
# YOLO11 + OpenCV
# ============================================================

# ------------------------------------------------------------
# 1. Install required libraries
# ------------------------------------------------------------
!pip install -q ultralytics opencv-python

# ------------------------------------------------------------
# 2. Import libraries
# ------------------------------------------------------------
from google.colab import files
from ultralytics import YOLO
from IPython.display import HTML, display
from base64 import b64encode

import cv2
import os

In [ ]:
import requests

video_url = "https://media.w3.org/2010/05/sintel/trailer.mp4"

r = requests.get(video_url)
r.raise_for_status()

with open("highway.mp4", "wb") as f:
    f.write(r.content)

print("Downloaded successfully!")
print("Size:", len(r.content) / (1024 * 1024), "MB")

In [ ]:
import requests

url = "https://raw.githubusercontent.com/kavyaguptaeng/Smart-Traffic-Management-System/master/highway.mp4"

r = requests.get(url, timeout=120)
r.raise_for_status()

with open("highway.mp4", "wb") as f:
    f.write(r.content)

print("Highway video downloaded successfully!")
print("Size:", len(r.content)/(1024*1024), "MB")

In [ ]:

# ============================================================
# 4. LOAD YOLO MODEL
# ============================================================

print("\nLoading YOLO11 model...")

model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully.")


# ============================================================
# 5. OPEN INPUT VIDEO
# ============================================================

input_video = "highway.mp4" # Corrected input video name
cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise Exception("Could not open uploaded video.")


# Read video properties

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print()
print("Video information")
print("--------------------------")
print("Width       :", width)
print("Height      :", height)
print("FPS         :", fps)
print("Total frames:", total_frames)


In [ ]:
# ============================================================
# 6. CREATE OUTPUT VIDEO
# ============================================================

output_video = "highway_annotated.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)


# ============================================================
# 7. VEHICLE CLASSES
# ============================================================

# COCO class IDs used by pretrained YOLO:
#
# 2 = car
# 3 = motorcycle
# 5 = bus
# 7 = truck

vehicle_classes = [2, 3, 5, 7]


In [ ]:
# ============================================================
# 8. PROCESS VIDEO FRAME BY FRAME
# ============================================================

frame_number = 0

print()
print("Starting vehicle detection...")
print()


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1


    # --------------------------------------------------------
    # YOLO VEHICLE DETECTION
    # --------------------------------------------------------

    results = model(
        frame,
        classes=vehicle_classes,
        conf=0.40,
        verbose=False
    )

    result = results[0]


    # Number of vehicles in current frame
    vehicle_count = 0


    # --------------------------------------------------------
    # PROCESS DETECTED OBJECTS
    # --------------------------------------------------------

    for box in result.boxes:

        vehicle_count += 1


        # Bounding box
        x1, y1, x2, y2 = map(
            int,
            box.xyxy[0].tolist()
        )


        # Confidence
        confidence = float(box.conf[0])


        # Class ID
        class_id = int(box.cls[0])


        # Class name
        class_name = model.names[class_id]


        # ----------------------------------------------------
        # DRAW BOUNDING BOX
        # ----------------------------------------------------

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            3
        )


        # ----------------------------------------------------
        # LABEL
        # ----------------------------------------------------

        label = f"{class_name} {confidence:.2f}"


        cv2.putText(
            frame,
            label,
            (x1, max(y1 - 10, 30)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )


    # --------------------------------------------------------
    # DISPLAY FRAME NUMBER
    # --------------------------------------------------------

    cv2.putText(
        frame,
        f"Frame: {frame_number}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )


    # --------------------------------------------------------
    # DISPLAY VEHICLE COUNT
    # --------------------------------------------------------

    cv2.putText(
        frame,
        f"Vehicles detected: {vehicle_count}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 0, 0),
        2
    )


    # --------------------------------------------------------
    # WRITE ANNOTATED FRAME
    # --------------------------------------------------------

    writer.write(frame)


    # Progress message every 50 frames

    if frame_number % 50 == 0:

        print(
            f"Processed {frame_number} / {total_frames} frames"
        )


# ============================================================
# 9. RELEASE VIDEO
# ============================================================

cap.release()

writer.release()


print()
print("==========================================")
print("Vehicle detection completed!")
print("==========================================")
print()
print("Output video:", output_video)



In [ ]:
# ============================================================
# 10. CONVERT VIDEO TO H264 FOR COLAB BROWSER DISPLAY
# ============================================================

print()
print("Preparing video for browser display...")


converted_video = "highway_annotated_colab.mp4"


!ffmpeg -y -loglevel error \
-i highway_annotated.mp4 \
-vcodec libx264 \
highway_annotated_colab.mp4


# ============================================================
# 11. DISPLAY VIDEO DIRECTLY IN COLAB
# ============================================================

print()
print("Annotated Highway Video")
print("=======================")


mp4 = open(converted_video, "rb").read()

video_data = "data:video/mp4;base64," + b64encode(mp4).decode()


display(
    HTML(
        f"""
        <video width="800"
               controls
               autoplay
               loop>

            <source src="{video_data}"
                    type="video/mp4">

        </video>
        """
    )
)


# ============================================================
# 12. DOWNLOAD OUTPUT VIDEO
# ============================================================

print()
print("You can also download the annotated video below:")

files.download(converted_video)